<img width="20%" alt="EarthDaily Analytics" src="https://raw.githubusercontent.com/earthdaily/Images/main/Corporate/EarthDaily.png" style="border-radius: 15%">

# 📘 EarthDaily Agriculture - In Season Monitoring Processor Bulk Extraction

## **✅ Step 1: Initialisation**

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

In [ ]:
from earthdaily.agriculture.services.workflow_manager import WorkflowManager
import os
manager = WorkflowManager("prod", log_to_console=True, log_level="DEBUG")

## **🛠️ Step 2: Get entities**

### Option 1 - Load entities from Earthdaily platform

In [ ]:
manager.load_seasonfields(crop_id="OTHERS")

print("\n🔎 First entity loaded:")
print(manager.sfd_list[['id', 'name']].head())

### Option 2 - Load entities from Earthdaily platform (Batch method - Recommended for large datasets)

In [ ]:
# This method uses batch processing to efficiently load 5000 entities

# Load all available entities using batch processing
manager.load_seasonfields_batch()

print("\n🔎 First entity loaded:")
print(manager.sfd_list[['id', 'name']].head())

### Option 3 -Load entities from file

In [ ]:
from earthdaily.agriculture.core.geometry import load_geodataframe

# Load entities from an external file (supports .shp, .parquet, .gpq, .geojson, .json, .gpkg, .csv)
file_path = "inputs/agroterrint.parquet"
manager.sfd_list = load_geodataframe(file_path, verbose=True)
print(manager.sfd_list.head())

## **📥 Step 3: Extract analytics - Debug function from processor_inseason_monitoring_functions.py**

### 🗺️ Configure extraction

In [ ]:
# Import your class
from earthdaily.agriculture.processors.processor_inseason_monitoring_functions import InSeasonMonitoringExtractor
extractor = InSeasonMonitoringExtractor(manager.bearer_token, manager.token_expiration,config=manager.config )

# Define column mapping to match DataFrame column names from the platform
column_mapping = {"crop": "crop.id"}

extractor.setup_inseason_monitoring_parameters(
                                    season_duration=120,
                                    season_start_day=1,
                                    season_start_month=4,
                                    year='2025',
                                    data_source='LR',
                                    partial_frequency=20,
                                    exclude_columns=[],
                                    column_mapping=column_mapping
)

### 🗺️ Test functions

In [ ]:
# Prepare test seasonfield_data
seasonfield_data = {
    "id": "z361x33",
    "geometry": "POLYGON ((-58.94540508 -13.72028589, -58.942163 -13.73172321, -58.928124090000004 -13.730314100000001, -58.93159922 -13.71888551, -58.94540508 -13.72028589))",
    "crop": "OTHERS"
}

#### Test get_inseason_monitoring_api

In [ ]:
print("\n--- Test: get_inseason_monitoring ---")
# Invoke the function
try:
    result = extractor.get_inseason_monitoring_api(seasonfield_data)
    print("✅ Raw API response received:")
    print(result if isinstance(result, dict) else result[:500])
except Exception as e:
    print(f"❌ API call failed: {e}")
    


#### Test get_inseason_monitoring_api_safe

In [ ]:
print("\n--- Test: get_inseason_monitoring_safe ---")
safe_result = extractor.get_inseason_monitoring_api_safe(seasonfield_data)
print(safe_result)

#### Test format_inseason_json

In [ ]:
print("\n--- Test: format_inseason_json ---")
if safe_result["success"] and safe_result["data"]:
    formatted_df = extractor.format_inseason_json(safe_result["data"])
    print(f"✅ Formatted DataFrame with {len(formatted_df)} rows:")
    display(formatted_df.head())
else:
    print("⚠️ Skipping format_inseason_json: No valid data from API.")

### 🗺️ process_single_entity

In [ ]:
import pandas as pd
row = pd.Series({
    "id": "z361x33",
    "geometry": "POLYGON ((-58.94540508 -13.72028589, -58.942163 -13.73172321, -58.928124090000004 -13.730314100000001, -58.93159922 -13.71888551, -58.94540508 -13.72028589))",
    "crop.id":"OTHERS"
})

result = extractor.process_single_entity_inseason_monitoring(row)

print(result)

### 🗺️ process_inseason_bulk_extraction_parallel

In [ ]:
top25 = manager.sfd_list.head(500)
print(top25.columns)
# Launch extraction with 10 threads 

result = extractor.process_inseason_bulk_extraction_parallel(
    entity_list=top25,
    params=None,
    max_workers=20,
    output_path=manager.output_result_dir,
    fail_safe=False,
    filter_column="crop.id",
    filter_value="CORN",
    filter_type="exclude", # filter type used to 'include' or 'exclude' row matching column and value filter
    generate_report=True
)

print(f"\nResults summary:")
print(f"Total: {result['total_calculations']}")
print(f"Success: {result['successful_calculations']}")
print(f"Failed: {result['failed_calculations']}")

print("\n🔍 First 3 errors:")
for i, error in enumerate(result['global_errors'][:3]):
    print(f"\nError {i+1}:")
    for key, value in error.items():
        print(f"  {key}: {value}")